# Variance Analysis Dashboard - Interactive Notebook

This notebook provides an interactive version of the Variance Analysis Dashboard with filtering capabilities, matrix visualizations, and statistical analysis.

## Features:
- Interactive data filtering by variance levels, stores, and cities
- Dynamic cohort-month matrix visualizations 
- Variance distribution analysis with box plots
- Summary statistics and key performance indicators
- Toggle between count and percentage views

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

In [ ]:
# Load and Preprocess Data
def load_data():
    """Load and preprocess the variance dashboard data."""
    # Load data with first row as headers
    df = pd.read_excel('dummy_data.xlsx', header=0)
    
    # Check if the actual headers are in the first row of data
    if df.iloc[0, 0] == 'MONTH':
        # Use the first row as column names and drop it
        df.columns = df.iloc[0]
        df = df.drop(df.index[0]).reset_index(drop=True)
        
        # Reset column names to remove any index references
        df.columns.name = None
    
    # Ensure numeric columns are properly typed
    numeric_columns = ['ORDER COUNT', 'CART SALES', 'DISCOUNT', 'NET REVENUE', 
                      'IDEAL FOOD COST', 'GROSS MARGIN', 'KITCHEN EBITDA', 'VARIANCE']
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Calculate variance percentage
    df['VARIANCE_PCT'] = (df['VARIANCE'] / df['NET REVENUE'] * 100).round(2)
    
    # Create variance categories
    df['VARIANCE_CATEGORY'] = df['VARIANCE_PCT'].apply(lambda x: 
        'High (>0.8%)' if x > 0.8 else 
        'Medium (0.4-0.8%)' if x > 0.4 else 
        'Low (<0.4%)')
    
    # Convert MONTH to datetime for better sorting
    df['MONTH_DATE'] = pd.to_datetime(df['MONTH'], format='%b-%Y')
    df = df.sort_values('MONTH_DATE')
    
    return df

# Load the data
df = load_data()
print(f"✅ Data loaded successfully!")
print(f"📊 Dataset shape: {df.shape}")
print(f"📅 Date range: {df['MONTH'].min()} to {df['MONTH'].max()}")
print(f"🏪 Total stores: {df['STORE'].nunique()}")
print(f"🏙️ Total cities: {df['CITY'].nunique()}")

# Display first few rows
display(df.head())

In [ ]:
# Create Data Filtering Functions
def apply_filters(df, variance_filter="All", variance_range=None, store="All", city="All"):
    """Apply selected filters to the dataframe."""
    filtered_df = df.copy()
    
    # Apply variance filter
    if variance_filter == "High Variance (>0.8%)":
        filtered_df = filtered_df[filtered_df['VARIANCE_PCT'] > 0.8]
    elif variance_filter == "Medium Variance (0.4-0.8%)":
        filtered_df = filtered_df[(filtered_df['VARIANCE_PCT'] >= 0.4) & (filtered_df['VARIANCE_PCT'] <= 0.8)]
    elif variance_filter == "Low Variance (<0.4%)":
        filtered_df = filtered_df[filtered_df['VARIANCE_PCT'] < 0.4]
    elif variance_filter == "Custom Range" and variance_range:
        min_var, max_var = variance_range
        filtered_df = filtered_df[(filtered_df['VARIANCE_PCT'] >= min_var) & (filtered_df['VARIANCE_PCT'] <= max_var)]
    
    # Apply other filters
    if store != 'All':
        filtered_df = filtered_df[filtered_df['STORE'] == store]
    
    if city != 'All':
        filtered_df = filtered_df[filtered_df['CITY'] == city]
    
    return filtered_df

def get_filter_options(df):
    """Get available options for filters."""
    variance_options = [
        "All",
        "High Variance (>0.8%)",
        "Medium Variance (0.4-0.8%)", 
        "Low Variance (<0.4%)",
        "Custom Range"
    ]
    
    store_options = ['All'] + sorted(df['STORE'].unique().tolist())
    city_options = ['All'] + sorted(df['CITY'].unique().tolist())
    
    matrix_options = [
        "REVENUE COHORT",
        "CM COHORT", 
        "EBITDA CATEGORY"
    ]
    
    return variance_options, store_options, city_options, matrix_options

print("✅ Data filtering functions created!")

In [ ]:
# Generate Summary Metrics
def create_summary_metrics(df):
    """Create and display summary metrics for variance analysis."""
    avg_variance = df['VARIANCE_PCT'].mean()
    total_stores = df['STORE'].nunique()
    high_variance_records = len(df[df['VARIANCE_PCT'] > 0.8])
    total_variance = df['VARIANCE'].sum()
    
    # Create metrics HTML
    metrics_html = f"""
    <div style="display: flex; gap: 20px; margin: 20px 0;">
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); 
                    color: white; padding: 20px; border-radius: 10px; text-align: center; flex: 1;">
            <h3 style="margin: 0; font-size: 2em;">{avg_variance:.2f}%</h3>
            <p style="margin: 5px 0 0 0;">Average Variance %</p>
        </div>
        <div style="background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%); 
                    color: white; padding: 20px; border-radius: 10px; text-align: center; flex: 1;">
            <h3 style="margin: 0; font-size: 2em;">{total_stores}</h3>
            <p style="margin: 5px 0 0 0;">Total Stores</p>
        </div>
        <div style="background: linear-gradient(135deg, #4facfe 0%, #00f2fe 100%); 
                    color: white; padding: 20px; border-radius: 10px; text-align: center; flex: 1;">
            <h3 style="margin: 0; font-size: 2em;">{high_variance_records}</h3>
            <p style="margin: 5px 0 0 0;">High Variance Records</p>
        </div>
        <div style="background: linear-gradient(135deg, #43e97b 0%, #38f9d7 100%); 
                    color: white; padding: 20px; border-radius: 10px; text-align: center; flex: 1;">
            <h3 style="margin: 0; font-size: 2em;">₹{total_variance:,.0f}</h3>
            <p style="margin: 5px 0 0 0;">Total Variance</p>
        </div>
    </div>
    """
    
    return metrics_html, {
        'avg_variance': avg_variance,
        'total_stores': total_stores, 
        'high_variance_records': high_variance_records,
        'total_variance': total_variance
    }

print("✅ Summary metrics function created!")

In [ ]:
# Build Interactive Widgets for Filters
# Get filter options
variance_options, store_options, city_options, matrix_options = get_filter_options(df)

# Create widgets
variance_filter = widgets.Dropdown(
    options=variance_options,
    value="All",
    description='Variance Filter:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

store_filter = widgets.Dropdown(
    options=store_options,
    value="All", 
    description='Store:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

city_filter = widgets.Dropdown(
    options=city_options,
    value="All",
    description='City:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

matrix_row = widgets.Dropdown(
    options=matrix_options,
    value="REVENUE COHORT",
    description='Matrix Rows:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

show_percentage = widgets.Checkbox(
    value=False,
    description='Show Count %',
    style={'description_width': 'initial'}
)

# Custom variance range slider (initially hidden)
variance_range_slider = widgets.FloatRangeSlider(
    value=[df['VARIANCE_PCT'].min(), df['VARIANCE_PCT'].max()],
    min=df['VARIANCE_PCT'].min(),
    max=df['VARIANCE_PCT'].max(),
    step=0.01,
    description='Variance Range:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px', display='none')
)

def on_variance_filter_change(change):
    if change['new'] == 'Custom Range':
        variance_range_slider.layout.display = 'block'
    else:
        variance_range_slider.layout.display = 'none'

variance_filter.observe(on_variance_filter_change, names='value')

# Group widgets
filter_box = widgets.VBox([
    widgets.HTML("<h3>🎛️ Dashboard Controls</h3>"),
    widgets.HBox([variance_filter, store_filter]),
    widgets.HBox([city_filter, matrix_row]),
    variance_range_slider,
    show_percentage
])

display(filter_box)
print("✅ Interactive widgets created!")

In [ ]:
# Create Cohort-Month Matrix Visualization
def create_cohort_month_matrix(df, matrix_row, show_percentage=False):
    """Create a matrix showing store counts by selected row attribute and month."""
    display_type = "Count %" if show_percentage else "Count"
    
    # Create pivot table
    pivot_table = df.groupby([matrix_row, 'MONTH']).size().reset_index(name='Store Count')
    pivot_matrix = pivot_table.pivot(index=matrix_row, columns='MONTH', values='Store Count').fillna(0)
    
    # Convert to percentage if requested
    if show_percentage:
        # Calculate percentage by column (month)
        pivot_matrix_pct = pivot_matrix.div(pivot_matrix.sum(axis=0), axis=1) * 100
        pivot_matrix_display = pivot_matrix_pct.round(1)
        color_scale_title = "Count %"
    else:
        pivot_matrix_display = pivot_matrix
        color_scale_title = "Store Count"
    
    # Reorder columns chronologically
    month_order = df.sort_values('MONTH_DATE')['MONTH'].unique()
    pivot_matrix_display = pivot_matrix_display.reindex(columns=month_order)
    
    # Create heatmap
    fig = px.imshow(
        pivot_matrix_display.values,
        x=pivot_matrix_display.columns,
        y=pivot_matrix_display.index,
        color_continuous_scale='Blues',
        title=f"Store {display_type} Heatmap by {matrix_row} and Month",
        labels={'color': color_scale_title},
        aspect="auto"
    )
    
    # Add text annotations
    for i, row in enumerate(pivot_matrix_display.index):
        for j, col in enumerate(pivot_matrix_display.columns):
            value = pivot_matrix_display.loc[row, col]
            if show_percentage:
                text = f"{value:.1f}%" if value > 0 else "0%"
            else:
                text = str(int(value))
            
            fig.add_annotation(
                x=j, y=i,
                text=text,
                showarrow=False,
                font=dict(color="white" if value > pivot_matrix_display.values.max()/2 else "black", size=10)
            )
    
    fig.update_layout(
        height=400,
        xaxis_title="Month",
        yaxis_title=matrix_row,
        title_x=0.5
    )
    
    return fig, pivot_matrix_display

print("✅ Cohort-month matrix function created!")

In [ ]:
# Generate Variance Analysis Charts
def create_variance_analysis(df, matrix_row):
    """Create variance analysis charts by selected row attribute."""
    
    # Box plot of variance by selected attribute
    fig_box = px.box(
        df, 
        x=matrix_row, 
        y='VARIANCE_PCT',
        title=f"Variance Distribution by {matrix_row}",
        labels={'VARIANCE_PCT': 'Variance %', matrix_row: matrix_row},
        color=matrix_row
    )
    fig_box.update_layout(height=400, title_x=0.5)
    fig_box.update_xaxes(tickangle=45)
    
    # Histogram of variance distribution
    fig_hist = px.histogram(
        df,
        x='VARIANCE_PCT',
        nbins=30,
        title="Overall Variance Distribution",
        labels={'VARIANCE_PCT': 'Variance %', 'count': 'Frequency'}
    )
    fig_hist.update_layout(height=400, title_x=0.5)
    
    # Scatter plot of variance vs net revenue
    fig_scatter = px.scatter(
        df,
        x='NET REVENUE',
        y='VARIANCE_PCT',
        color=matrix_row,
        size='ORDER COUNT',
        hover_data=['STORE', 'MONTH'],
        title="Variance % vs Net Revenue",
        labels={'NET REVENUE': 'Net Revenue (₹)', 'VARIANCE_PCT': 'Variance %'}
    )
    fig_scatter.update_layout(height=400, title_x=0.5)
    
    return fig_box, fig_hist, fig_scatter

print("✅ Variance analysis chart functions created!")

In [ ]:
# Display Summary Statistics Tables
def create_statistics_table(df, matrix_row):
    """Generate summary statistics table for variance analysis."""
    stats_df = df.groupby(matrix_row)['VARIANCE_PCT'].agg([
        'count', 'mean', 'median', 'min', 'max', 'std'
    ]).round(2)
    stats_df.columns = ['Count', 'Mean %', 'Median %', 'Min %', 'Max %', 'Std Dev %']
    
    # Add styled formatting
    styled_stats = stats_df.style.format({
        'Mean %': '{:.2f}%',
        'Median %': '{:.2f}%', 
        'Min %': '{:.2f}%',
        'Max %': '{:.2f}%',
        'Std Dev %': '{:.2f}%'
    }).background_gradient(subset=['Mean %'], cmap='RdYlBu_r')
    
    return stats_df, styled_stats

def create_detailed_summary(df):
    """Create a detailed summary of the dataset."""
    summary_data = {
        'Metric': [
            'Total Records',
            'Date Range', 
            'Unique Stores',
            'Unique Cities',
            'Average Variance %',
            'High Variance Records (>0.8%)',
            'Medium Variance Records (0.4-0.8%)',
            'Low Variance Records (<0.4%)',
            'Total Revenue (₹)',
            'Total Variance (₹)'
        ],
        'Value': [
            f"{len(df):,}",
            f"{df['MONTH'].min()} to {df['MONTH'].max()}",
            f"{df['STORE'].nunique():,}",
            f"{df['CITY'].nunique():,}",
            f"{df['VARIANCE_PCT'].mean():.2f}%",
            f"{len(df[df['VARIANCE_PCT'] > 0.8]):,}",
            f"{len(df[(df['VARIANCE_PCT'] >= 0.4) & (df['VARIANCE_PCT'] <= 0.8)]):,}",
            f"{len(df[df['VARIANCE_PCT'] < 0.4]):,}",
            f"₹{df['NET REVENUE'].sum():,.0f}",
            f"₹{df['VARIANCE'].sum():,.0f}"
        ]
    }
    
    summary_df = pd.DataFrame(summary_data)
    return summary_df

print("✅ Statistics table functions created!")

In [ ]:
# Interactive Dashboard
output = widgets.Output()

def update_dashboard(*args):
    """Update the dashboard based on widget values."""
    with output:
        clear_output(wait=True)
        
        # Get current filter values
        variance_val = variance_filter.value
        store_val = store_filter.value
        city_val = city_filter.value
        matrix_val = matrix_row.value
        percentage_val = show_percentage.value
        
        # Apply filters
        if variance_val == "Custom Range":
            variance_range_val = variance_range_slider.value
            filtered_df = apply_filters(df, variance_val, variance_range_val, store_val, city_val)
        else:
            filtered_df = apply_filters(df, variance_val, None, store_val, city_val)
        
        # Check if data exists
        if len(filtered_df) == 0:
            display(HTML("<h3 style='color: red;'>⚠️ No data matches the selected filters.</h3>"))
            return
        
        # Display summary metrics
        metrics_html, metrics_data = create_summary_metrics(filtered_df)
        display(HTML(f"<h2>📊 Key Performance Indicators</h2>{metrics_html}"))
        
        # Display matrix heatmap
        fig_matrix, matrix_data = create_cohort_month_matrix(filtered_df, matrix_val, percentage_val)
        display(HTML(f"<h2>🔥 Store {'Count %' if percentage_val else 'Count'} Matrix</h2>"))
        fig_matrix.show()
        
        # Display matrix table
        display(HTML(f"<h3>📋 Matrix Data Table</h3>"))
        display(matrix_data)
        
        # Display variance analysis charts
        display(HTML(f"<h2>📈 Variance Analysis Charts</h2>"))
        fig_box, fig_hist, fig_scatter = create_variance_analysis(filtered_df, matrix_val)
        
        fig_box.show()
        fig_hist.show()
        fig_scatter.show()
        
        # Display statistics table
        stats_df, styled_stats = create_statistics_table(filtered_df, matrix_val)
        display(HTML(f"<h3>📊 Variance Statistics by {matrix_val}</h3>"))
        display(styled_stats)
        
        # Display detailed summary
        summary_df = create_detailed_summary(filtered_df)
        display(HTML(f"<h3>📋 Detailed Summary</h3>"))
        display(summary_df)

# Attach observers to widgets
variance_filter.observe(update_dashboard, names='value')
store_filter.observe(update_dashboard, names='value')
city_filter.observe(update_dashboard, names='value')
matrix_row.observe(update_dashboard, names='value')
show_percentage.observe(update_dashboard, names='value')
variance_range_slider.observe(update_dashboard, names='value')

# Initial dashboard update
update_dashboard()

# Display output
display(output)

print("✅ Interactive dashboard created! Use the controls above to filter and explore the data.")